# GenAI Pipeline — Testing Notebook

LLM-based screening of publications for alternative protein relevance and pillar classification.
Uses Claude with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [2]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

DB_PATH = "../publications.db"
OUTPUT_DIR = Path(".")


### 2. Data Inspection

In [3]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df()) # Check available tables

df = con.sql("SELECT * FROM publications_raw").df()
con.close()

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nScope distribution:")
print(df["scope"].value_counts())
print(f"\nPillar distribution:")
print(df["pillar"].value_counts())
df.head()

Tables:
               name
0  publications_raw

Shape: (4815, 7)

Columns: ['id', 'title', 'abstract', 'year', 'scope', 'pillar', 'research_category']

Scope distribution:
scope
out    3773
in     1042
Name: count, dtype: int64

Pillar distribution:
pillar
NA    3773
PB     695
F      151
CC      98
CM      98
Name: count, dtype: int64


,id,title,abstract,year,scope,pillar,research_category
0,pub.1192791591,Mushroom: an emerging source for next generati...,"Background: In recent years, plant-based and a...",2025,in,PB,Ingredient optimisation
1,pub.1187762379,Plant-Based Alternatives to Meat Products,Animal proteins have been used in the formulat...,2025,in,PB,Other
2,pub.1193391974,Influence of processing on protein quality and...,While meat is an established source of high-qu...,2025,in,PB,Ingredient optimisation
3,pub.1188899046,Solid State Fermentation—A Promising Approach ...,The increasing demand for sustainable dietary ...,2025,in,F,Other
4,pub.1193352421,"Physicochemical, Microbiological and Sensory E...",The bioactive properties of a phenolic extract...,2025,in,PB,End product formulation


### 3. Balanced Subset Creation

Two test sets with balanced representation across scope and pillar:
- `initial_test_data`: 14 records (6 out, 2 PB, 2 F, 2 cultivated, 2 cross-cutting)
- `test_data_100`: 110 records (40 out, 30 PB, 15 F, 15 cultivated, 10 cross-cutting)


In [4]:
def create_balanced_sample(df, n_out, n_pb, n_f, n_cultivated, n_cross, random_state=123): # Can change random_state for different samples - must be integer for reproducibility
    out_sample = df[df["scope"] == "out"].sample(n=n_out, random_state=random_state)
    pb_sample = df[df["pillar"] == "PB"].sample(n=n_pb, random_state=random_state)
    f_sample = df[df["pillar"] == "F"].sample(n=n_f, random_state=random_state)
    cult_sample = df[df["pillar"] == "CM"].sample(n=n_cultivated, random_state=random_state)
    cross_sample = df[df["pillar"] == "CC"].sample(n=n_cross, random_state=random_state)
    combined = pd.concat(
        [out_sample, pb_sample, f_sample, cult_sample, cross_sample], ignore_index=True
    )
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [5]:
initial_test_data = create_balanced_sample(df, n_out=6, n_pb=2, n_f=2, n_cultivated=2, n_cross=2)
print(f"initial_test_data: {initial_test_data.shape}")
# initial_test_data[["id", "title", "scope", "pillar"]]

initial_test_data: (14, 7)


In [6]:
PILLAR_ORDER = ["PB", "F", "CM", "CC", "NA"]

initial_test_data_review = initial_test_data.copy()
initial_test_data_review["pillar"] = pd.Categorical(
    initial_test_data_review["pillar"], categories=PILLAR_ORDER, ordered=True
)
initial_test_data_review.sort_values("pillar")[["id", "title", "scope", "pillar", "abstract"]]

,id,title,scope,pillar,abstract
0,pub.1191456334,"Comparative Analysis of Composition, Texture, ...",in,PB,The increasing demand for plant-based foods ha...
10,pub.1196236950,The influence of phytase on the solubility of ...,in,PB,Rapeseed (Brassica napus subsp. napus) protein...
5,pub.1185503350,From a Coriander Mayonnaise to a Vegan Analogu...,in,F,"History aside, traditional mayonnaise faces a ..."
6,pub.1192499872,Microalgal biomass in the European food indust...,in,F,The growing need for sustainable and nutrient-...
1,pub.1193443783,Proliferation and metabolic activity of the At...,in,CM,Introduction: Fish cell lines represent a powe...
7,pub.1192676912,Consumer Trust in Emerging Food Technologies: ...,in,CM,Consumer trust plays a critical role in the su...
11,pub.1196060665,Nutrient Equivalence of Plant-Based and Cultur...,in,CC,Meat provides high-quality protein and essenti...
13,pub.1193813897,The Potential of Fermentation-Based Processing...,in,CC,Proteins are fundamental to food systems due t...
2,pub.1184066957,Screening of Plant UDP-Glycosyltransferases fo...,out,NA,To cover the rising demand for natural food dy...
3,pub.1185271226,Construction of a CRISPR‐Cas9‐Based Genetic Ed...,out,NA,Putrescine plays a significant role in green f...


In [7]:
test_data_100 = create_balanced_sample(df, n_out=40, n_pb=30, n_f=15, n_cultivated=15, n_cross=10)
print(f"test_data_100: {test_data_100.shape}")
print(test_data_100[["scope", "pillar"]].value_counts())

test_data_100: (110, 7)
scope  pillar
out    NA        40
in     PB        30
       CM        15
       F         15
       CC        10
Name: count, dtype: int64


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [8]:
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(initial_test_data, "initial_test_data.xlsx")
save_subset(test_data_100, "test_data_100.xlsx")

Saved 14 records to initial_test_data.xlsx
Saved 110 records to test_data_100.xlsx


### 5. Load Prompt and Select Dataset

In [51]:
PROMPT_PATH = "1_prompt_debugging/v5/prompt_v5.md"

# ← CHANGE THIS to switch between datasets
# Options: initial_test_data (14 records), test_data_100 (110 records), incorrect_scope_data (ONLY for re-run after testing and review)
DATASET = incorrect_scope_data

In [52]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food science research.

Your task is to classify a research publication based on its title and abstract. First, determine whether the publication concerns research on alternative proteins intended for use in human food, applying the definitions, boundaries, and exclusions below. Provide a confidence score reflecting your certainty in the scope decision. Then, if in scope, assign the relevant alternative protein category label(s).

General Definition
Alternative proteins are plant-based, fermentation-derived, or cultivated proteins, ingredients, and products, intended as alternatives to conventional animal proteins in the human diet - including meat, seafood, dairy, and egg analogues, as well as novel ingredients such as microbial proteins. Publications are in scope if they fall into either or both of the following two categories:
- Studies on the societal, policy, regulatory, consumer acceptance, environmental, life-cycle, or techno-economi

### 6. API Call with Prompt Caching

In [ ]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["haiku"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

class _ClassificationBase(BaseModel):
    scope: Literal["in", "out"]
    confidence: int = Field(ge=1, le=5)
    plant_based: bool
    fermentation: bool
    cultivated: bool

class _ClassificationWithReasoning(_ClassificationBase):
    reasoning: str

ClassificationSchema = _ClassificationWithReasoning if INCLUDE_REASONING else _ClassificationBase


client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[ # Cache control only works for prompts of 1024+ tokens, so not currently useful. Batch API will be later though.
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


In [58]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [59]:
def classify_with_error_handling(row, system_prompt):
    pub_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pub_id}: model returned no structured output")
                return {"id": pub_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()}
            output["id"] = pub_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pub_id}: {last_error}")
    return {"id": pub_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [60]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data

In [61]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Run 1 / 1
  [1/17] pub.1193443783
  [2/17] pub.1189767013
  [3/17] pub.1193123540
  [4/17] pub.1186250605
  [5/17] pub.1184676351
  [6/17] pub.1185271743
  [7/17] pub.1190672497
  [8/17] pub.1192751374
  [9/17] pub.1184367118
  [10/17] pub.1193249719
  [11/17] pub.1192827486
  [12/17] pub.1186661415
  [13/17] pub.1191677996
  [14/17] pub.1196064670
  [15/17] pub.1184344359
  [16/17] pub.1183660277
  [17/17] pub.1186213546

Completed: 17 records across 1 run(s)
Successful: 17
Errors: 0


,scope_LLM,confidence_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,reasoning_LLM,id,status,run
0,in,4,False,False,True,This study investigates serum reduction and gr...,pub.1193443783,ok,1
1,in,4,False,False,True,The review focuses on alginate extraction and ...,pub.1189767013,ok,1
2,in,4,True,False,False,This agronomic study focuses on pea and faba b...,pub.1193123540,ok,1
3,in,4,False,False,True,The study explicitly includes cultivated meat ...,pub.1186250605,ok,1
4,in,4,False,True,False,The study uses fermentation of microalgae (Chl...,pub.1184676351,ok,1
5,in,5,False,False,False,This publication explicitly addresses policy i...,pub.1185271743,ok,1
6,in,4,False,False,True,This consumer tracking report explicitly colle...,pub.1190672497,ok,1
7,out,2,False,False,False,While this publication concerns protein extrac...,pub.1192751374,ok,1
8,in,4,True,False,False,The study explicitly investigates consumer pra...,pub.1184367118,ok,1
9,out,2,False,True,False,Although this study involves fungal biomass (A...,pub.1193249719,ok,1


In [62]:
# Compare LLM predictions against ground truth labels
result_cols = ["id", "run", "scope_LLM", "confidence_LLM", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.insert(result_cols.index("confidence_LLM") + 1, "reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "scope", "pillar"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison["correct_scope"] = comparison["scope"] == comparison["scope_LLM"]

# Derive pillar_LLM from boolean flags:
# single True → that pillar; multiple True → CC (cross-cutting); none True → NA
comparison["pillar_LLM"] = comparison[["plant_based_LLM", "fermentation_LLM", "cultivated_LLM"]].apply(
    lambda r: "CC" if r.sum() > 1
              else "PB" if r["plant_based_LLM"]
              else "F"  if r["fermentation_LLM"]
              else "CM" if r["cultivated_LLM"]
              else "NA",
    axis=1
)
comparison["correct_pillar"] = comparison["pillar"] == comparison["pillar_LLM"]

# Summary metrics
print(f"Scope accuracy ({REPETITIONS} run(s)):  {comparison['correct_scope'].mean():.0%}  (n={len(comparison)})")
print(f"Pillar accuracy ({REPETITIONS} run(s)): {comparison['correct_pillar'].mean():.0%}  (n={len(comparison)})")

both_in = (comparison["scope"] == "in") & (comparison["scope_LLM"] == "in")
print(f"Pillar accuracy — both in-scope:   {comparison.loc[both_in, 'correct_pillar'].mean():.0%}  (n={both_in.sum()})")

if REPETITIONS > 1:
    per_run = comparison.groupby("run")[["correct_scope", "correct_pillar"]].mean()
    print(f"\nPer-run accuracy:\n{per_run.to_string()}")

display_cols = ["id", "title", "abstract", "scope", "scope_LLM", "confidence_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["correct_scope", "pillar", "pillar_LLM", "correct_pillar", "plant_based_LLM", "fermentation_LLM", "cultivated_LLM"]
comparison[display_cols]


Scope accuracy (1 run(s)):  65%  (n=17)
Pillar accuracy (1 run(s)): 59%  (n=17)
Pillar accuracy — both in-scope:   82%  (n=11)


,id,title,abstract,scope,scope_LLM,confidence_LLM,reasoning_LLM,correct_scope,pillar,pillar_LLM,correct_pillar,plant_based_LLM,fermentation_LLM,cultivated_LLM
0,pub.1193443783,Proliferation and metabolic activity of the At...,Introduction: Fish cell lines represent a powe...,in,in,4,This study investigates serum reduction and gr...,True,CM,CM,True,False,False,True
1,pub.1189767013,Keeping an eye on alginate: innovations and op...,"Alginate is a biocompatible, biodegradable, br...",in,in,4,The review focuses on alginate extraction and ...,True,CM,CM,True,False,False,True
2,pub.1193123540,Intercropping traditional pea varieties with f...,"Although being a historical farming practise, ...",in,in,4,This agronomic study focuses on pea and faba b...,True,PB,PB,True,True,False,False
3,pub.1186250605,Changing Food in a Changing World: Assessing C...,"Background/Objectives: In recent decades, the ...",in,in,4,The study explicitly includes cultivated meat ...,True,CM,CM,True,False,False,True
4,pub.1184676351,Fermentation of a wine pomace and microalgae b...,This study aimed to generate new functional in...,in,in,4,The study uses fermentation of microalgae (Chl...,True,CC,F,False,False,True,False
5,pub.1185271743,Toward consumer-focused food policies: a toolb...,In transitioning toward consuming more sustain...,in,in,5,This publication explicitly addresses policy i...,True,CC,NA,False,False,False,False
6,pub.1190672497,Consumer Insights Tracker End of Year Report (...,This report presents the findings from the Con...,in,in,4,This consumer tracking report explicitly colle...,True,CM,CM,True,False,False,True
7,pub.1192751374,Processing a maize germ side stream: de-oiling...,Maize grain processing usually starts with deg...,in,out,2,While this publication concerns protein extrac...,False,PB,NA,False,False,False,False
8,pub.1184367118,We are a family! Exploring flexitarian househo...,It is widely accepted that the (over)consumpti...,in,in,4,The study explicitly investigates consumer pra...,True,PB,PB,True,True,False,False
9,pub.1193249719,Evaluating Scale-Up Cultivation Modes for Aspe...,Organic-waste-derived volatile fatty acids (VF...,in,out,2,Although this study involves fungal biomass (A...,False,F,F,True,False,True,False


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign scope and pillar, I need to save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in the 1_prompt_debugging folder.
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results. Includes both metrics and 
6. Write a text document about v1 results and what changes you want to make to the prompt. Repeat from step 1.

In [63]:
save_dir = Path("1_prompt_debugging/v5") # CHANGE ME
save_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([
    {"metric": "scope_accuracy",                "value": f"{comparison['correct_scope'].mean():.0%}",               "n": len(comparison)},
    {"metric": "pillar_accuracy",               "value": f"{comparison['correct_pillar'].mean():.0%}",              "n": len(comparison)},
    {"metric": "pillar_accuracy_both_in_scope", "value": f"{comparison.loc[both_in, 'correct_pillar'].mean():.0%}", "n": int(both_in.sum())},
])

with pd.ExcelWriter(save_dir / f"{MODEL}_results.xlsx") as writer:
    comparison[display_cols].to_excel(writer, sheet_name="comparison", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

print(f"Saved to {save_dir / f'{MODEL}_results.xlsx'}")


Saved to 1_prompt_debugging\v5\claude-sonnet-4-6_results.xlsx


In [41]:
# Create dataset of only rows where LLM got scope wrong, for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["correct_scope"], "id"]
incorrect_scope_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_scope_data


,id,title,abstract,year,scope,pillar,research_category
0,pub.1193443783,Proliferation and metabolic activity of the At...,Introduction: Fish cell lines represent a powe...,2025,in,CM,Cell culture media
1,pub.1189767013,Keeping an eye on alginate: innovations and op...,"Alginate is a biocompatible, biodegradable, br...",2025,in,CM,Scaffolding
2,pub.1193123540,Intercropping traditional pea varieties with f...,"Although being a historical farming practise, ...",2025,in,PB,Crop development
3,pub.1186250605,Changing Food in a Changing World: Assessing C...,"Background/Objectives: In recent decades, the ...",2025,in,CM,Consumer & market research
4,pub.1184676351,Fermentation of a wine pomace and microalgae b...,This study aimed to generate new functional in...,2025,in,CC,Ingredient optimisation
5,pub.1185271743,Toward consumer-focused food policies: a toolb...,In transitioning toward consuming more sustain...,2025,in,CC,Other
6,pub.1190672497,Consumer Insights Tracker End of Year Report (...,This report presents the findings from the Con...,2025,in,CM,Consumer & market research
7,pub.1192751374,Processing a maize germ side stream: de-oiling...,Maize grain processing usually starts with deg...,2025,in,PB,Ingredient optimisation
8,pub.1184367118,We are a family! Exploring flexitarian househo...,It is widely accepted that the (over)consumpti...,2025,in,PB,Consumer & market research
9,pub.1193249719,Evaluating Scale-Up Cultivation Modes for Aspe...,Organic-waste-derived volatile fatty acids (VF...,2025,in,F,Feedstocks
